In [4]:
from bs4 import BeautifulSoup
import re
import tiktoken

# Load HTML (normally from a file or variable; here you paste your string)
with open("index.html", "r", encoding="utf-8") as f:
    html_content = f.read()

soup = BeautifulSoup(html_content, "html.parser")
articles = soup.find_all("li", class_="article-item")

enc = tiktoken.encoding_for_model("gpt-3.5-turbo")

title_token_count = 0
title_word_count = 0
abstract_token_count = 0
abstract_word_count = 0

for art in articles:
    title = art.get("data-title", "").strip()
    abstract = art.get("data-abstract", "").strip()

    # Count words
    title_word_count += len(title.split())
    abstract_word_count += len(abstract.split())

    # Count tokens
    title_token_count += len(enc.encode(title))
    abstract_token_count += len(enc.encode(abstract))

summary = {
    "Total Articles": len(articles),
    "Title Word Count": title_word_count,
    "Title Token Count": title_token_count,
    "Abstract Word Count": abstract_word_count,
    "Abstract Token Count": abstract_token_count,
}

summary


{'Total Articles': 7424,
 'Title Word Count': 93924,
 'Title Token Count': 144468,
 'Abstract Word Count': 634392,
 'Abstract Token Count': 915946}

In [7]:
import random
from bs4 import BeautifulSoup

def extract_random_articles_from_html(html_path, n_articles=10):
    with open(html_path, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f, 'html.parser')

    # Find all journals
    journal_sections = soup.find_all("div", class_="accordion-item")

    all_articles = []

    for section in journal_sections:
        journal_name = section.find("h3", class_="journal-header").text.strip().replace("–", "-").replace("—", "-")
        articles = section.find_all("li", class_="article-item")
        for article in articles:
            title = article.get("data-title", "").strip()
            abstract = article.get("data-abstract", "").strip()
            authors = article.get("data-authors", "").strip()
            doi_tag = article.find("a", class_="read-more-link")
            doi = doi_tag['href'] if doi_tag else "N/A"

            all_articles.append({
                "journal": journal_name,
                "title": title,
                "abstract": abstract,
                "authors": authors,
                "doi": doi
            })

    # Shuffle and pick random articles from distinct journals
    random.shuffle(all_articles)
    selected = {}
    for art in all_articles:
        if len(selected) >= n_articles:
            break
        journal = art['journal']
        if journal not in selected:
            selected[journal] = art

    return list(selected.values())

# Example usage
if __name__ == "__main__":
    selected_articles = extract_random_articles_from_html("index.html", n_articles=10)
    for i, art in enumerate(selected_articles, 1):
        #print(f"\nArticle {i}:")
        #print(f"Journal:  {art['journal']}")
        print(f"Article {i} - Title:    {art['title']}")
        #print(f"Authors:  {art['authors']}")
        #print(f"DOI:      {art['doi']}")
        #print(f"Abstract: {art['abstract'][:300]}...")


Article 1 - Title:    mitochondria transported by kinesin-3 prevent localized calcium spiking to inhibit caspase-dependent specialized cell death
Article 2 - Title:    moving beyond the motor cortex: a brain-wide evaluation of target locations for intracranial speech neuroprostheses
Article 3 - Title:    abscisic acid modulation of drought tolerance and essential oil biosynthesis in lavandula coronopifolia poir
Article 4 - Title:    does hyperbaric oxygen therapy chambers safe for the operator’s cardiovascular health: an occupational safety issue(perspective)
Article 5 - Title:    setd2 suppresses tumorigenesis in a krasg12c-driven lung cancer model, and its catalytic activity is regulated by histone acetylation
Article 6 - Title:    copii component sec13 is required for peripheral myelination and schwann cell maintenance
Article 7 - Title:    india tests new tools to predict local monsoon floods
Article 8 - Title:    conformal integration of multifunctional nanomembranes on fibers tow

In [31]:
import requests

def ask_ollama(prompt, model='mistral'):
    url = 'http://localhost:11434/api/generate'
    response = requests.post(url, json={
        'model': model,
        'prompt': prompt,
        'stream': False
    })
    return response.json()

# Example usage
response = ask_ollama("hi")
print(response)

{'model': 'mistral', 'created_at': '2025-10-02T21:16:07.056289Z', 'response': " Hello! How can I assist you today? Is there something specific you would like to know or discuss? I'm here to help with a wide range of topics, from answering questions and providing information, to helping generate ideas, and even just chatting about various subjects. Let me know what you need!", 'done': True, 'done_reason': 'stop', 'context': [3, 29473, 12782, 4, 29473, 23325, 29576, 2370, 1309, 1083, 6799, 1136, 3922, 29572, 2459, 1504, 2313, 3716, 1136, 1450, 1505, 1066, 1641, 1210, 4110, 29572, 1083, 29510, 29487, 2004, 1066, 2084, 1163, 1032, 6103, 3587, 1070, 14585, 29493, 1245, 25170, 4992, 1072, 8269, 2639, 29493, 1066, 9306, 9038, 6534, 29493, 1072, 1787, 1544, 1252, 15526, 1452, 4886, 15343, 29491, 3937, 1296, 1641, 1535, 1136, 1695, 29576], 'total_duration': 23054282234, 'load_duration': 3416276033, 'prompt_eval_count': 5, 'prompt_eval_duration': 1156097385, 'eval_count': 63, 'eval_duration': 18

In [20]:
!ollama serve

Error: listen tcp 127.0.0.1:11434: bind: address already in use


In [23]:
!ps aux | grep ollama

morning+   61703  0.0  0.0 1930512 31880 pts/2   Tl   22:34   0:00 ollama run mistral
ollama     64555  0.8  0.0 2004240 32164 ?       Ssl  22:51   0:00 /usr/local/bin/ollama serve
morning+   64581 83.3  0.0 231928  3720 pts/3    Ss+  22:51   0:00 /bin/bash -c ps aux | grep ollama
morning+   64583  0.0  0.0 231252  2392 pts/3    S+   22:51   0:00 grep ollama


In [18]:
!sudo pkill ollama

[sudo] password for morningrise: 
